In [32]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression , LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score , accuracy_score ,classification_report

In [33]:
df = pd.read_csv('../data/processed/dollar_price_features.csv', index_col=0, parse_dates=True)
df.index.name = 'Data'
df.head()

,Open,Low,High,Close,DayOfWeek,Is_Imputed,Daily_Return,Close_Lag_1,Close_Lag_3,Close_Lag_7,MA_7,MA_30
Data,,,,,,,,,,,,
2012-01-25,19000.0,19000.0,19000.0,19000.0,Wednesday,False,-0.095238,21000.0,19000.0,18200.0,19228.571429,16959.666667
2012-01-26,17000.0,17000.0,17000.0,17000.0,Thursday,False,-0.105263,19000.0,21000.0,18200.0,19342.857143,17084.666667
2012-01-27,17000.0,17000.0,17000.0,17000.0,Friday,True,0.000000,17000.0,21000.0,18200.0,19171.428571,17146.333333
2012-01-28,17700.0,17700.0,17700.0,17700.0,Saturday,False,0.041176,17000.0,19000.0,19000.0,19000.000000,17206.333333
2012-01-29,18300.0,18300.0,18300.0,18300.0,Sunday,False,0.033898,17700.0,17000.0,19000.0,18814.285714,17289.000000


# تقسیم زمانی (Chronological Split)


In [34]:
split_data = df.index[int(len(df) * 0.8)]
print('تاریخ تقسیم', split_data)

train = df[df.index < split_data]
test = df[df.index >= split_data]

print('تعداد Train :', len(train))
print('تعدداد Test :', len(test))

تاریخ تقسیم 2021-01-04 00:00:00
تعداد Train : 3204
تعدداد Test : 802


جدا کردن X و y

In [35]:
feature_cols = ['Close_Lag_1', 'Close_Lag_3', 'Close_Lag_7', 'MA_7', 'MA_30']

X_train = train[feature_cols]
y_train = train['Close']

X_test = test[feature_cols]
y_test = test['Close']

In [36]:
X_train.shape , X_test.shape

((3204, 5), (802, 5))

# Linear Regression

In [37]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [38]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(((y_test - y_pred) ** 2).mean())
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae:,.0f}')
print(f'RMSE: {rmse:,.0f}')
print(f'R²: {r2:.4f}')

MAE: 2,514
RMSE: 5,984
R²: 0.9961


مدل Naive

In [39]:
naive_pred = X_test['Close_Lag_1']

mae_naive = mean_absolute_error(y_test, naive_pred)
rmse_naive = np.sqrt(((y_test - naive_pred) ** 2).mean())
r2_naive = r2_score(y_test, naive_pred)

print(f"MAE (Naive): {mae_naive:,.0f}")
print(f"RMSE (Naive): {rmse_naive:,.0f}")
print(f"R² (Naive): {r2_naive:.4f}")

MAE (Naive): 2,406
RMSE (Naive): 5,977
R² (Naive): 0.9961


 # تغییر هدف مدل‌سازی

In [40]:
df['Target_Return'] = df['Daily_Return'].shift(-1)
df = df.dropna(subset=['Target_Return'])

تقسیم زمانی دوباره

In [41]:
split_data = df.index[int(len(df) * 0.8)]

train = df[df.index < split_data]
test = df[df.index >= split_data]

feature_cols = ['Close_Lag_1', 'Close_Lag_3', 'Close_Lag_7', 'MA_7', 'MA_30', 'Daily_Return']

X_train = train[feature_cols]
y_train = train['Target_Return']

X_test = test[feature_cols]
y_test = test['Target_Return']

آموزش و ارزیابی

In [42]:
model_return = LinearRegression()
model_return.fit(X_train, y_train)

y_pred_return = model_return.predict(X_test)

r2_return = r2_score(y_test, y_pred_return)
mae_return = mean_absolute_error(y_test, y_pred_return)

print(f"R²: {r2_return:.4f}")
print(f"MAE: {mae_return:.6f}")

R²: -0.0512
MAE: 0.008118


In [43]:
naive_return_pred = [y_train.mean()] * len(y_test)

r2_naive_return = r2_score(y_test, naive_return_pred)
print(f"R² (پیش‌بینی میانگین ثابت): {r2_naive_return:.4f}")

R² (پیش‌بینی میانگین ثابت): -0.0000


# ساخت هدف جدید

Classification

In [44]:
df['Target_Direction'] = (df['Daily_Return'].shift(-1) > 0).astype(int)
df = df.dropna(subset=['Target_Direction'])

بررسی توازن کلاس‌ها

In [45]:
df['Target_Direction'].value_counts(normalize=True)

Target_Direction
0    0.605743
1    0.394257
Name: proportion, dtype: float64

تقسیم زمانی و آماده‌سازی X/y

In [46]:
split_data = df.index[int(len(df) * 0.8)]

train = df[df.index < split_data]
test = df[df.index >= split_data]

feature_cols = ['Close_Lag_1', 'Close_Lag_3', 'Close_Lag_7', 'MA_7', 'MA_30', 'Daily_Return']

X_train = train[feature_cols]
y_train = train['Target_Direction']

X_test = test[feature_cols]
y_test = test['Target_Direction']

مدل Classification — Logistic Regression

In [47]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred_clf = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_clf):.4f}")
print(classification_report(y_test, y_pred_clf))

Accuracy: 0.5755
              precision    recall  f1-score   support

           0       0.59      0.94      0.73       481
           1       0.26      0.03      0.06       320

    accuracy                           0.58       801
   macro avg       0.43      0.48      0.39       801
weighted avg       0.46      0.58      0.46       801



In [48]:
baseline_pred = [0] * len(y_test)
print(f"Accuracy (Baseline - همیشه بگو 'پایین/ثابت'): {accuracy_score(y_test, baseline_pred):.4f}")

Accuracy (Baseline - همیشه بگو 'پایین/ثابت'): 0.6005
